# 01 Datasets and per-cell features

Builds the C2 tables (`earth_features`, `moon_discs`, `mars_discs`, `targets`) with the columns and dtypes of `doppelgrund.fixtures.synthetic`.
`REAL_DATA = False` (default locally) loads the synthetic tables so the notebook runs offline. `REAL_DATA = True` (default on Colab) builds them from the real datasets: Earth Engine for Earth terrain, thermal, land cover and aridity; local files (downloaded by `fetch`, section 1) for everything else.
**Colab**: upload `doppelgrund_colab.zip` (`python notebooks/make_colab_zip.py` writes it to `data/interim/`) to `MyDrive/DoppelGrund/`, set `EE_PROJECT` in the first code cell, run all. The about 4 GB of raw files and all outputs persist in `MyDrive/DoppelGrund/raw` and `MyDrive/DoppelGrund/out`; Earth Engine batches and Diviner bands are checkpointed, so after a disconnect re-running resumes. **Not yet run end to end: the Earth Engine reducers (no credentials where this was written) and the full 18-band Diviner download.**
Locally, outputs go to `data/interim/notebooks/` only. Web bundles are never touched. Dataset IDs below are a proposal: `config/datasets.yaml` is human-owned.

In [ ]:
import sys, subprocess, importlib.util, zipfile, json, re, warnings
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/DoppelGrund")  # upload doppelgrund_colab.zip here; raw downloads and outputs persist here
    pip = [p for p, m in [("h3", "h3"), ("rasterio", "rasterio"), ("geopandas", "geopandas"), ("earthengine-api", "ee")] if importlib.util.find_spec(m) is None]
    if pip: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pip], check=True)
    if not Path("/content/DoppelGrund/pipeline").exists(): zipfile.ZipFile(ROOT / "doppelgrund_colab.zip").extractall("/content/DoppelGrund")
    sys.path.insert(0, "/content/DoppelGrund/pipeline")  # not `pip install -e`: pyproject pins would fight Colab's numpy/pandas

import numpy as np, pandas as pd
from doppelgrund.config import REPO, EXPECTED_FEATURES, load_profiles, load_crosswalk, load_landmarks
from doppelgrund.fixtures.synthetic import (make_synthetic, select_land_cells, fibonacci_lattice, targets_table, haversine_km,
                                     load_land, LITH_COLS, N_DISCS, EARTH_R_KM)

REAL_DATA = IN_COLAB      # False: synthetic tables, offline. True: real data (Earth Engine + downloaded files)
EE_PROJECT = ""           # Google Cloud project with Earth Engine enabled (real data only)
N_DISCS_LIMIT = None      # smoke test: only N evenly spaced discs and N evenly spaced hexes
FETCH_DIVINER = True      # False: skip the 2.8 GB Diviner download; Moon thermal stays NaN outside bands already reduced
DATA = ROOT / "raw" if IN_COLAB else REPO / "data/raw"
OUT = ROOT / "out" if IN_COLAB else REPO / "data/interim/notebooks"
WORK = Path("/content/work") if IN_COLAB else REPO / "data/interim/_work"  # scratch: unzipped permafrost, Diviner tables in flight
for p in (OUT, WORK): p.mkdir(parents=True, exist_ok=True)
prof = load_profiles()
SEED, R_KM = prof.params["seed"], prof.params["footprint_radius_km"]
BODY_R_M = {"moon": 1_737_400.0, "mars": 3_396_190.0}
SCALE_M = 470  # stage-1 baseline for every body (LDEM_64 ~474 m, MOLA 463 m, Earth resampled)

## 1. Source register

Status from the 2026-10-31 access check (`datasets.md`) updated with the 2026-11-10 download check (`data/raw/CHECK.md`, per-dataset `data/raw/*/SOURCE.md`). **UNVERIFIED** means ID, path, version or licence was not confirmed on a fetched page.

In [ ]:
S = lambda body, name, ref, status, feeds: dict(body=body, dataset=name, ref=ref, status=status, feeds=feeds)
SOURCES = pd.DataFrame([
 S("earth","Copernicus GLO-30 DEM","ee:COPERNICUS/DEM/GLO30_2024_1 (band DEM; old GLO30 superseded)","VERIFIED (licence wording changed, check PDF)","slope_*, relief_std_m, frac_slope_lt_10deg"),
 S("earth","MODIS Terra LST","ee:MODIS/061/MOD11A2 (LST_Day_1km, LST_Night_1km, x0.02 K)","VERIFIED","lst_mean_K, diurnal_range_K, annual_range_K, frac_below_0C"),
 S("earth","MODIS land cover","ee:MODIS/061/MCD12Q1 LC_Type1 (16 barren, 15 snow/ice), 2010-2019","VERIFIED","barren_frac, snow_ice_frac, mask"),
 S("earth","MODIS NDVI","ee:MODIS/061/MOD13A2 NDVI (not in access check)","UNVERIFIED","ndvi_mean"),
 S("earth","ASTER GED AG100","ee:NASA/ASTER_GED/AG100_003 (emissivity_band10-14, x0.001)","VERIFIED (index definition below is a placeholder)","aster_mafic_idx"),
 S("earth","TerraClimate","ee:IDAHO_EPSCOR/TERRACLIMATE (pr, pet x0.1)","VERIFIED","aridity_index = P/PET"),
 S("earth","GLiM 0.5 deg","PANGAEA doi:10.1594/PANGAEA.788537, 720x360 ASCII grid, 16 classes (2-letter codes in Classnames.txt)","VERIFIED download (CC-BY-3.0); long class names are from the paper, not the download","lith_*"),
 S("earth","Permafrost (Obu et al.)","PANGAEA doi:10.1594/PANGAEA.888600, UiO_PEX_PERPROB_5.0, 1 km, EPSG:3995, north of about 25 N","VERIFIED download (CC-BY-3.0)","permafrost_prob (NH only)"),
 S("earth","GVP Holocene volcanoes","volcano.si.edu WFS GeoJSON, 1214 points","downloaded; version and licence UNVERIFIED (site returned 403)","tag_volcanic"),
 S("earth","Natural Earth land 1:110m","config/ne_land.geojson","VERIFIED","land cells"),
 S("moon","LOLA LDEM_64","PDS LRO-L-LOLA-4-GDR-V1.0, 23040x11520 int16, 64 ppd, height = 0.5 m x DN","VERIFIED download (licence not stated)","terrain"),
 S("moon","Diviner GCP","PDS LRO-L-DLRE-5-GCP-V1.0, 18 tables of 0.5 deg x 0.25 h bins, -9999 missing, re-downloaded and reduced here","VERIFIED (sha256 per table)","thermal"),
 S("moon","Lunar Prospector GRS FeO","PDS LP-L-GRS-5-ELEM-ABUNDANCE-V1.0, 2 deg equal-area pixels, W_FEO g/g (true resolution about 5 deg)","VERIFIED download (licence not stated); coarse, mare vs highland only","lith proxy (FeO)"),
 S("mars","MOLA 463 m DEM","USGS Astropedia GeoTIFF, 46080x23040 int16, m above the areoid","VERIFIED download (CC0)","terrain"),
 S("mars","TES thermal inertia (Putzig 2005)","tes.asu.edu ti_nmap2003.bin + LASP nmsk2003.bin mask (Wayback)","VERIFIED download (licence not stated); TI, not a temperature","side column tes_ti, not scored"),
 S("mars","Geologic map of Mars (Tanaka 2014)","USGS SIM 3292 shapefile, 1311 polygons, 44 units, 1:20M","VERIFIED download (cite authors); no mineral map was downloaded","lith proxy via a PROPOSED unit mapping"),
])
SOURCES[["body","dataset","status"]]

### Downloads

`FILES` and `GCP_SHA` hold every URL and sha256 this notebook needs (copied from `data/raw/*/SOURCE.md`). `fetch` downloads a file only if it is missing and verifies the hash, so re-running is cheap. Locally `data/raw` already holds everything except the 18 Diviner tables; on Colab it all lands in `MyDrive/DoppelGrund/raw`. The SIM 3292 database zip (809 MB, mostly a hillshade) is fetched to scratch only to extract the geology shapefile, then deleted.

In [ ]:
import hashlib, os, requests

PDS = "https://pds-geosciences.wustl.edu"
FILES = {  # key: (url, sha256, path under DATA)
    "glim": ("https://epic.awi.de/31092/1/hartmann-moosdorf_2012.zip", "43b4ce3276b155d804db8ff9fb227d620b4c35015a4cf564eac4d06d2b69d88e", "earth_glim/hartmann-moosdorf_2012.zip"),
    "perprob": ("https://store.pangaea.de/Publications/ObuJ-etal_2018/UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.zip", "d19338ada0c188eb99e1a06eeeccc8e4055f037aeff96590f61f2f427c3cac5d", "earth_permafrost/UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.zip"),
    "gvp": ("https://webservices.volcano.si.edu/geoserver/GVP-VOTW/ows?service=WFS&version=2.0.0&request=GetFeature&typeName=GVP-VOTW:Smithsonian_VOTW_Holocene_Volcanoes&outputFormat=application/json", None, "earth_volcanoes/Smithsonian_VOTW_Holocene_Volcanoes.geojson"),  # live database, hash not stable
    "ldem": (PDS + "/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/data/lola_gdr/cylindrical/img/ldem_64.img", "1c4958699d4cffd7e777d51421b6044bba300aa0bef286a544309efb082cdbd6", "moon_dem_stage1/ldem_64.img"),
    "lpgrs": (PDS + "/lunar/lp-l-grs-5-elem-abundance-v1/lp_9001/data/lpgrs_high1_elem_abundance_2deg.tab", "e866953499813ec089067f1985dfb92fe7adc8957a2121bb115a3b27ea14e946", "moon_feo/lpgrs_high1_elem_abundance_2deg.tab"),
    "mola": ("https://planetarymaps.usgs.gov/mosaic/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif", "38a4eb0b4452855b8dabfac40a367b458555ab4c01b31235807ad0a53c031f4c", "mars_dem/Mars_MGS_MOLA_DEM_mosaic_global_463m.tif"),
    "tes_ti": ("http://tes.asu.edu/mineral_map/ti_nmap2003.bin", "ce28ab69365023cf3d85d8a12d9758f38d61f77c9b0af7580b5fbc186c3e1379", "mars_tes_ti/ti_nmap2003.bin"),
    "tes_mask": ("http://web.archive.org/web/20070208014625id_/http://lasp.colorado.edu:80/inertia/nmsk2003.bin", "4d3ceb3160c83fa51eee464955b59aa0cc7933e66e540bbaa10cfd2b5f789ec9", "mars_tes_ti/nmsk2003.bin"),
    "sim3292": ("https://pubs.usgs.gov/sim/3292/downloads/sim3292_database.zip", "ef75f059b1f2f5a47aff55e6c6e39500cf1ab65e743fabc5d463b97bc10b6541", None),  # zip only; shapefile extracted below
}
GCP_SHA = {  # band -> sha256 of the 156 MB raw table (data/raw/moon_diviner/SOURCE.md)
    "80n90n": "b8f5e30354e086bf0c5e3559c35f05aed0f3de887a2792dcf07a9b0f66837e5b",
    "70n80n": "82d6f73429410e08a8fdb4f41dab44cf278129ed8e47d8f0dd6667102defefb6",
    "60n70n": "2191949f8618239793b6d412fa937e2438386ec098f9adc09a82546c519bd0f8",
    "50n60n": "4f1c2a43dd7e61e715df560aaebd400e950e7edc1cc195dc62ff6c86d7b8b4a6",
    "40n50n": "5377ddbdfe226c216288c1116c20845fdf101a462847108ad887f11176b992e2",
    "30n40n": "5c69b39ef0173d1fb282b3e7e74ac1a5a43cd9c1dd956299f2c778715204cea3",
    "20n30n": "e15ea704b50adf16c25210dbc1669b965eb636fa81c90828778c853e8c2e21aa",
    "10n20n": "9c3fc386357f082f0c4b9b00e64682475478cf9449430832da684384d1f2c4b5",
    "00n10n": "d333fc13b605beff4d977bd78299d3dd4c98905ea9008f69f4ee45073d7bedea",
    "10s00s": "b47dd7ee7fe90ade46e376b4aa62bf153e8a9f2aeecd3593a32f3a514d61d981",
    "20s10s": "321033b7550795d8be70627c192f40779c5de757dacb42b36ff15e7d62854071",
    "30s20s": "08ec2886b74e4ce80096f983b4b352ee51defa2ab3089a060c2407bfe26c5a3b",
    "40s30s": "12c422698db88de3e9156e4c1aa4c94b2be97678cac2b05444caccc4fd32c344",
    "50s40s": "973609411fe7ab82f52f9b57c3a7d3f23753273a5934a92c1656b81505f5f5c4",
    "60s50s": "f050506f56618075a296f7544e10a5fd199d5d5fa82bc3a40053a99943f1d097",
    "70s60s": "c8c0ae10968961740a5fe80cddc97856ffeb1f83821587d0b6fe2114658c7294",
    "80s70s": "c6c9b3a84e1e9972237b07fd8dd1c0f44899ab8ffe9a403a7528ef7cd38b1cda",
    "90s80s": "4bc7093fe5bf08ca7fd0c7ddaf94f7c452b27f705fe975a73c8614d8d41e28b6",
}
GCP_BANDS = list(GCP_SHA)
GCP_URL = PDS + "/lro/urn-nasa-pds-lro_diviner_derived1/data_derived_gcp/global_cumul_avg_cyl_{}_002.tab"

def fetch(key, url, sha256, dest):
    """Download url to dest unless it exists; verify sha256 (None = live endpoint, hash only printed). Written as .part then renamed, so an existing file is complete."""
    dest = Path(dest)
    if dest.exists(): return dest
    dest.parent.mkdir(parents=True, exist_ok=True); part = dest.with_name(dest.name + ".part"); h = hashlib.sha256()
    with requests.get(url, stream=True, timeout=60, headers={"User-Agent": "doppelgrund-notebook"}) as r, open(part, "wb") as f:
        r.raise_for_status()
        for chunk in r.iter_content(1 << 20): f.write(chunk); h.update(chunk)
    if sha256 and h.hexdigest() != sha256:
        part.unlink(); raise ValueError(f"{key}: sha256 {h.hexdigest()} != expected {sha256}")
    os.replace(part, dest); print(f"fetched {key}: {dest.stat().st_size / 1e6:.0f} MB, sha256 {h.hexdigest()[:12]}")
    return dest

get = lambda key: fetch(key, *FILES[key][:2], DATA / FILES[key][2])

def mars_shapefile():
    """SIM 3292 geology shapefile under DATA; downloads the 809 MB zip to WORK only if it is missing, extracts the Geology members, deletes the zip."""
    d = DATA / "mars_geology/SIM3292_MarsGlobalGeologicGIS_20M/SIM3292_Shapefiles"; shp = d / "SIM3292_Global_Geology.shp"
    if not shp.exists():
        z = fetch("sim3292", *FILES["sim3292"][:2], WORK / "sim3292_database.zip")
        with zipfile.ZipFile(z) as zf:
            for n in zf.namelist():
                if "/SIM3292_Shapefiles/SIM3292_Global_Geology." in n and n.rsplit(".", 1)[-1] in ("shp", "shx", "dbf", "prj"):
                    d.mkdir(parents=True, exist_ok=True); (d / n.rsplit("/", 1)[-1]).write_bytes(zf.read(n))
        z.unlink()
    return shp

## 2. Horn slope (one algorithm, all bodies)

Same 3x3 Horn operator on a height grid for every body (Zhou & Liu 2004; Grohmann 2017 preprint). Grid spacing is passed in metres per axis, so it also handles the east-west shrink of equirectangular rasters. Slope statistics are only comparable at the same baseline (Kreslavsky & Head 2000; Rosenburg et al. 2011): every value here is labelled with `SCALE_M` = 470 m. Resampling GLO30 by area-mean does **not** reproduce true 470 m-grid roughness (Wang & Wu 2017, search-result only), which `02_harmonisation` treats as an uncertainty.

In [ ]:
def horn_slope_deg(z, dx, dy):
    """Horn 3x3 slope in degrees; z (m), dx/dy grid spacing (m). Edge ring is NaN."""
    p = np.pad(z.astype(np.float64), 1, mode="edge")
    a,b,c,d,f,g,h,i = (p[:-2,:-2],p[:-2,1:-1],p[:-2,2:],p[1:-1,:-2],p[1:-1,2:],p[2:,:-2],p[2:,1:-1],p[2:,2:])
    dzdx = ((c + 2*f + i) - (a + 2*d + g)) / (8*dx)
    dzdy = ((g + 2*h + i) - (a + 2*b + c)) / (8*dy)
    s = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
    s[[0,-1],:] = np.nan; s[:,[0,-1]] = np.nan
    return s

def terrain_stats(z, slope, mask):
    s = slope[mask & np.isfinite(slope)]
    if s.size < 9: return dict.fromkeys(EXPECTED_FEATURES[:4], np.nan)
    return dict(slope_p50_deg=np.percentile(s,50), slope_p90_deg=np.percentile(s,90),
                relief_std_m=float(np.nanstd(z[mask])), frac_slope_lt_10deg=float(np.mean(s<10)))

# self-check: a 10 degree plane gives 10 degrees
yy, xx = np.mgrid[0:20,0:20]; assert abs(np.nanmedian(horn_slope_deg(np.tan(np.radians(10))*470*xx, 470, 470)) - 10) < 1e-6

## 3. Earth: H3 res-4 land cells

Land cells come from `select_land_cells()` (Natural Earth, cell centre on land; about 80k). Two sources per hex:

**Earth Engine reducers** (`earth_remote`, all per hex polygon, batches checkpointed to `OUT/ee_batches/`):
- **Terrain**: GLO30 `DEM` mosaic (`COPERNICUS/DEM/GLO30_2024_1`), `reduceResolution(mean)` then `reproject` to 470 m (area-mean, never nearest), Horn slope by explicit kernels (same operator as above), `percentile([50,90])` of slope, mean elevation and mean elevation squared (`relief_std_m` = sqrt of the difference), mean of `slope<10`.
- **Thermal**: MOD11A2 2010-2019. `lst_mean_K` = mean of (day+night)/2; `diurnal_range_K` = day minus night, i.e. the **Terra 10:30 / 22:30** overpass pair; `annual_range_K` = max minus min of monthly means; `frac_below_0C` = fraction of 8-day composites below 273.15 K. Typical MODIS LST error on barren ground is about 1 K (Wan 2014).
- **Mask inputs**: MCD12Q1 2010-2019 barren (16) and snow/ice (15) fractions; NDVI. **Composition index**: ASTER GED silicate/mafic index (placeholder formula). **Aridity**: TerraClimate P/PET.

**Local files** (`earth_local`, sampled at the hex centre, `ponytail:` upgrade = zonal mean over the hex): GLiM 0.5 deg class mapped through the crosswalk, Obu permafrost probability (north of about 25 N, NaN elsewhere by design), `tag_volcanic` (a GVP Holocene volcano within 25 km of the centre) and `land_frac` (7-point stencil on the Natural Earth land polygon).

Earth Engine on Colab: `ee.Authenticate()` opens a link, then `ee.Initialize(project=EE_PROJECT)`. Reducer output names are checked on a Sahara probe hex before any batch runs, so a wrong name fails in seconds, not hours.

In [ ]:
# Reducer outputs we need back, by the names the reducers give them (z_mean and z2_mean become relief_std_m)
EE_RAW = ["slope_p50_deg", "slope_p90_deg", "z_mean", "z2_mean", "frac_slope_lt_10deg", "lst_mean_K", "diurnal_range_K", "annual_range_K",
          "frac_below_0C", "barren_frac", "snow_ice_frac", "ndvi_mean", "aster_mafic_idx", "aridity_index"]
EE_COLS = [c for c in EE_RAW if c not in ("z_mean", "z2_mean")] + ["relief_std_m"]

def earth_remote(cells, batch=2000):
    """Earth Engine reducers per hex. Finished batches are kept in OUT/ee_batches/NNNN.parquet and skipped on a re-run (Colab disconnect). NOT RUN here: no Earth Engine credentials."""
    import ee, h3  # imported here so the offline run needs no earthengine-api
    if not EE_PROJECT: raise ValueError("set EE_PROJECT (a Google Cloud project with Earth Engine enabled) in the first code cell")
    try: ee.Initialize(project=EE_PROJECT)
    except Exception: ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
    dem = (ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1").select("DEM").mosaic()
           .setDefaultProjection("EPSG:4326", None, 30))
    z = dem.reduceResolution(ee.Reducer.mean(), maxPixels=1024).reproject("EPSG:4326", None, SCALE_M)
    kx = ee.Kernel.fixed(3, 3, [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], normalize=False)
    ky = ee.Kernel.fixed(3, 3, [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], normalize=False)
    # ponytail: 'scale' is metres at the equator; EW shrink ignored, so poleward slopes are biased. Reproject for |lat|>60.
    gx = z.convolve(kx).divide(8 * SCALE_M); gy = z.convolve(ky).divide(8 * SCALE_M)
    slope = gx.hypot(gy).atan().multiply(180 / np.pi).rename("slope")
    terrain = (z.rename("z_mean").addBands(z.pow(2).rename("z2_mean")).addBands(slope.lt(10).rename("frac_slope_lt_10deg")))
    lst = ee.ImageCollection("MODIS/061/MOD11A2").filterDate("2010-01-01", "2020-01-01")
    day = lst.select("LST_Day_1km").mean().multiply(0.02); night = lst.select("LST_Night_1km").mean().multiply(0.02)
    mean_ = lst.map(lambda i: i.select("LST_Day_1km").add(i.select("LST_Night_1km")).multiply(0.01).rename("m"))
    months = ee.ImageCollection([mean_.filter(ee.Filter.calendarRange(m, m, "month")).mean() for m in range(1, 13)])
    thermal = (day.add(night).divide(2).rename("lst_mean_K").addBands(day.subtract(night).rename("diurnal_range_K"))
               .addBands(months.max().subtract(months.min()).rename("annual_range_K"))
               .addBands(mean_.map(lambda i: i.lt(273.15)).mean().rename("frac_below_0C")))
    lc = ee.ImageCollection("MODIS/061/MCD12Q1").filterDate("2010-01-01", "2020-01-01").select("LC_Type1")
    cover = lc.map(lambda i: i.eq(16)).mean().rename("barren_frac").addBands(lc.map(lambda i: i.eq(15)).mean().rename("snow_ice_frac"))
    ndvi = ee.ImageCollection("MODIS/061/MOD13A2").filterDate("2010-01-01", "2020-01-01").select("NDVI").mean().multiply(0.0001).rename("ndvi_mean")
    ged = ee.Image("NASA/ASTER_GED/AG100_003")
    e = lambda n: ged.select(f"emissivity_band{n}")
    mafic = e(13).add(e(14)).subtract(e(10)).subtract(e(11)).multiply(0.001).rename("aster_mafic_idx")  # UNVERIFIED placeholder index
    tc = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE").filterDate("2010-01-01", "2020-01-01")
    arid = tc.select("pr").sum().divide(tc.select("pet").sum().multiply(0.1)).rename("aridity_index")  # same 10-yr window in numerator and denominator; pet scaled x0.1
    # single-band images keep the reducer's own output name (p50, p90, mean), multi-band images keep band names; hence the renames
    groups = [(slope, ee.Reducer.percentile([50, 90]), SCALE_M, {"p50": "slope_p50_deg", "p90": "slope_p90_deg"}),
              (terrain, ee.Reducer.mean(), SCALE_M, {}), (thermal, ee.Reducer.mean(), 1000, {}),
              (cover.addBands(ndvi), ee.Reducer.mean(), 500, {}),
              (mafic, ee.Reducer.mean(), 100, {"mean": "aster_mafic_idx"}), (arid, ee.Reducer.mean(), 4638, {"mean": "aridity_index"})]

    def reduce(hexes):
        fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Polygon([[p[1], p[0]] for p in h3.cell_to_boundary(c)]), {"h3": c}) for c in hexes])
        got = [pd.DataFrame([f["properties"] for f in img.reduceRegions(fc, red, sc).getInfo()["features"]]).set_index("h3").rename(columns=ren)
               for img, red, sc, ren in groups]
        return pd.concat(got, axis=1)

    def finish(df):
        df = df.copy(); df["relief_std_m"] = np.sqrt((df["z2_mean"] - df["z_mean"] ** 2).clip(lower=0))
        return df.reindex(columns=EE_COLS)  # a batch where a dataset has no pixels at all (e.g. ASTER in Antarctica) gets NaN, not a missing column

    probe = reduce([h3.latlng_to_cell(23.0, 10.0, 4)])  # a Sahara hex: every dataset covers it, so a missing column means a wrong output name
    miss = [c for c in EE_RAW if c not in probe.columns]
    if miss: raise KeyError(f"Earth Engine reducer output lacks {miss}; got {list(probe.columns)}")
    print(finish(probe).T.round(3))
    ck = OUT / "ee_batches"; ck.mkdir(exist_ok=True); parts = []; nb = -(-len(cells) // batch)
    for k in range(0, len(cells), batch):
        f = ck / f"{k // batch:04d}.parquet"; hx = list(cells[k:k + batch])
        if f.exists() and set(pd.read_parquet(f).index) == set(hx): parts.append(pd.read_parquet(f)); continue  # finished before a disconnect
        part = finish(reduce(hx)).reindex(hx); part.to_parquet(f); parts.append(part); print(f"batch {k // batch + 1}/{nb} done")
    return pd.concat(parts).rename_axis("h3").reset_index()

In [ ]:
O = ["mafic_volcanic", "felsic_feldspathic", "carbonate", "siliciclastic", "evaporite", "ice"]
# GLiM first-level class -> ontology fractions (rows sum to 1, or all 0 for nd/wb). PROPOSAL for human review: config/ontology_crosswalk.csv is human-owned
GLIM = {"vb": [1,0,0,0,0,0], "pb": [1,0,0,0,0,0], "py": [.5,.5,0,0,0,0], "va": [0,1,0,0,0,0], "vi": [0,1,0,0,0,0],
        "pa": [0,1,0,0,0,0], "pi": [0,1,0,0,0,0], "mt": [0,.5,0,.5,0,0], "sc": [0,0,1,0,0,0], "ss": [0,0,0,1,0,0],
        "su": [0,0,0,1,0,0], "sm": [0,0,.3,.7,0,0], "ev": [0,0,0,0,1,0], "ig": [0,0,0,0,0,1], "nd": [0]*6, "wb": [0]*6}
GLIM_W = {**dict.fromkeys(GLIM, 1.0), "su": 0.5, "sm": 0.5, "mt": 0.5, "py": 0.5}  # low weight for unconsolidated/mixed (used when a cell holds several classes)
xw = load_crosswalk()

def crosswalk(source, codes, proposal):
    """Ontology fractions (n x 6, columns LITH_COLS) for unit codes. Rows of config/ontology_crosswalk.csv with this `source` win (human-owned); else `proposal`.
    Codes not in the table and all-zero rows (water, no data) give NaN."""
    rows = xw[xw["source"].str.lower() == source]
    table = rows.set_index("unit_code")[O].astype(float) if len(rows) else proposal
    out = np.array(table.reindex(list(codes)), float)
    out[~(np.nansum(out, axis=1) > 0)] = np.nan
    return pd.DataFrame(out, columns=LITH_COLS)

xyz = lambda la, lo: np.c_[np.cos(np.radians(la)) * np.cos(np.radians(lo)), np.cos(np.radians(la)) * np.sin(np.radians(lo)), np.sin(np.radians(la))]

def earth_local(cells):
    """GLiM lithology, permafrost, volcano tag and land fraction at the hex centres, from files under DATA.
    ponytail: centre sampling (GLiM 0.5 deg cell, permafrost 1 km pixel); upgrade = zonal mean over the hex polygon."""
    import h3, rasterio, shapely
    from pyproj import Transformer
    from scipy.spatial import cKDTree
    ll = np.array([h3.cell_to_latlng(c) for c in cells]).reshape(-1, 2); lat, lon = ll[:, 0], ll[:, 1]
    # GLiM: row 0 = +90, col 0 = -180, no .prj (EPSG:4326); value -> 2-letter code through Classnames.txt
    with zipfile.ZipFile(get("glim")) as zf:
        grid = np.loadtxt(zf.open("glim_wgs84_0point5deg.txt.asc"), skiprows=6, dtype=int)
        names = pd.read_csv(zf.open("Classnames.txt"), sep=";").set_index("Value_")["xx"]
    v = grid[np.clip(((90 - lat) / 0.5).astype(int), 0, 359), np.clip(((lon + 180) / 0.5).astype(int), 0, 719)]
    lith = crosswalk("glim", pd.Series(v).map(names), pd.DataFrame(GLIM, index=O).T)  # -9999 (no polygon) -> NaN
    # permafrost probability: unzip the 1.9 GB GeoTIFF to scratch once (reading inside the zip is too slow), polar stereographic EPSG:3995
    with zipfile.ZipFile(get("perprob")) as zf:
        tif = next(n for n in zf.namelist() if n.endswith(".tif"))
        if not (WORK / tif).exists(): zf.extract(tif, WORK)
    perma = np.full(len(cells), np.nan)
    with rasterio.open(WORK / tif) as ds:
        x, y = Transformer.from_crs(4326, ds.crs.to_wkt(), always_xy=True).transform(lon, lat)
        col, row = (x - ds.transform.c) / ds.transform.a, (y - ds.transform.f) / ds.transform.e
        ok = (col >= 0) & (col < ds.width) & (row >= 0) & (row < ds.height)  # outside the grid (south of about 25 N) stays NaN
        band = ds.read(1)
        perma[ok] = band[row[ok].astype(int), col[ok].astype(int)]
    perma = np.where((perma >= 0) & (lat >= 25), perma, np.nan)  # nodata is -3.4e38 (and NaN); the product's domain is north of 25 N (grid corners below hold 0, not data)
    # GVP: any Holocene volcano within 25 km of the centre (chord on the unit sphere)
    vol = json.loads(get("gvp").read_text(encoding="utf-8"))["features"]
    vll = np.array([f["geometry"]["coordinates"][:2] for f in vol])
    d, _ = cKDTree(xyz(vll[:, 1], vll[:, 0])).query(xyz(lat, lon))
    # land fraction: centre + 6 points 60% of the way to each vertex, inside the Natural Earth land polygon. ponytail: 7-point stencil; upgrade = polygon intersection area
    bnd = [list(h3.cell_to_boundary(c)) for c in cells]
    vtx = np.array([b + [b[0]] * (6 - len(b)) for b in bnd]).reshape(len(cells), 6, 2)  # pentagons repeat a vertex
    s_lat = lat[:, None] + 0.6 * (vtx[..., 0] - lat[:, None])
    s_lon = lon[:, None] + 0.6 * ((vtx[..., 1] - lon[:, None] + 180) % 360 - 180)
    land = load_land(); ins = shapely.contains_xy(land, ((s_lon + 180) % 360 - 180).ravel(), s_lat.ravel()).reshape(-1, 6).sum(1)
    land_frac = (ins + shapely.contains_xy(land, lon, lat)) / 7
    out = pd.DataFrame({"h3": list(cells), "permafrost_prob": perma, "tag_volcanic": d <= 2 * np.sin(25 / (2 * EARTH_R_KM)),
                        "land_frac": land_frac, "lat": lat, "lng": lon})
    return pd.concat([out, lith], axis=1)

## 4. Moon and Mars discs (r = 23.7 km)

Disc centres are the same near-uniform Fibonacci lattice as the synthetic tables (3000 discs per body, longitude -180..180). Terrain: window of the LDEM_64 / MOLA raster around the centre, same Horn slope with metric grid spacing, statistics inside the 23.7 km circle; `vf_dem` is the fraction of non-nodata pixels in the circle. **LDEM_64**: int16, 23040 x 11520, column 0 = 0 deg E, height = 0.5 m x DN above the 1737.4 km sphere (GDAL returns radius, so the raw DN is used). **MOLA**: int16 metres above the areoid, 46080 x 23040, column 0 = -180 deg E.

**Moon thermal** is re-reduced from the 18 Diviner GCP tables (0.5 deg x 0.25 h bins; the original 0.5 deg derived GeoTIFF lost local time, so it cannot give an overpass-matched range). Per 0.5 deg bin: `T(10:30)` = mean of the two 0.25 h bins either side of 10.5 h (centres 10.375 and 10.625), `T(22:30)` likewise, `lst_mean_K` = mean over the observed local-time bins, `frac_below_0C` = fraction of observed bins below 273.15 K (a diurnal-cycle fraction, not Earth's time-series fraction, so a **proxy that differs by body**). `diurnal_range_K = T(10:30) - T(22:30)`, matching MODIS Terra. A bin with no observation at either time stays NaN (never imputed); the disc value is the NaN-mean over the 3x3 bins around the centre (about the 47 km disc). `annual_range_K` stays NaN. Each band is downloaded, reduced, checkpointed in `OUT/gcp_lt/` and deleted. Polar PRP products are not used (so cold traps are not resolved).
**Mars thermal**: no temperature product was downloaded (TES thermal inertia is not a surface temperature and is **not** converted), so thermal columns are NaN. TES TI at the disc centre is kept as a side column `tes_ti` (not scored) with `tes_ti_measured` (1 = uninterpolated TES data, 0 = interpolated fill).
**Composition proxies, different quantities per body**: Moon = Lunar Prospector GRS FeO (2 deg equal-area pixels, nearest pixel to the disc centre; true resolution about 5 deg, so mare vs highland only), linear 6-16 wt% to mafic vs feldspathic. Mars = Tanaka 2014 geologic unit at the disc centre, mapped to the six classes by a **PROPOSAL for human review** (below, by unit suffix); `config/ontology_crosswalk.csv` rows with source `mars_tanaka2014` override it.

In [ ]:
def disc_terrain(dem, ppd, lat, lon, body, lon0=0.0, scale=1.0, r_km=R_KM):
    """Terrain stats for one disc. dem: int16 DN (rows, cols), row 0 = +90, col 0 = lon0 deg E; height = DN * scale; -32768 = nodata.
    ponytail: polar discs use a capped lon window (the 2 discs nearest each pole are mildly distorted)."""
    R = BODY_R_M[body]; dth = np.radians(1.0 / ppd); dy = R * dth
    dx = max(R * np.cos(np.radians(lat)) * dth, dy * 0.05)
    hw_r = int(np.ceil(r_km * 1000 / dy)) + 1; hw_c = min(int(np.ceil(r_km * 1000 / dx)) + 1, 2000)
    r0 = int((90 - lat) * ppd); c0 = int(((lon - lon0) % 360) * ppd)
    rows = np.clip(np.arange(r0 - hw_r, r0 + hw_r + 1), 0, dem.shape[0] - 1)
    cols = np.arange(c0 - hw_c, c0 + hw_c + 1) % dem.shape[1]
    raw = dem[np.ix_(rows, cols)]
    z = np.where(raw == -32768, np.nan, raw * scale).astype(np.float64)
    ry, rx = np.mgrid[-hw_r:hw_r+1, -hw_c:hw_c+1]
    mask = np.hypot(ry * dy, rx * dx) <= r_km * 1000
    return {**terrain_stats(z, horn_slope_deg(z, dx, dy), mask), "vf_dem": float(np.isfinite(z[mask]).mean())}

def gcp_reduce(tab):
    """One 10-degree Diviner GCP table -> (first lat row, (4, n_lat, 720) float32): T(10:30), T(22:30), mean T, fraction < 273.15 K, per 0.5 deg bin.
    Table rows: clon, clat, ltim (bin centre, h), ..., tbol (col 10); -9999 = no observation."""
    d = pd.read_csv(tab, usecols=[0, 1, 2, 10], skipinitialspace=True)
    ix = np.floor((d.iloc[:, 0].to_numpy() + 180) / 0.5).astype(int); iy = np.floor((90 - d.iloc[:, 1].to_numpy()) / 0.5).astype(int)
    k = np.floor(d.iloc[:, 2].to_numpy() / 0.25).astype(int)  # centres 0.125, 0.375, ... -> bins 0..95
    assert ix.min() >= 0 and ix.max() < 720 and k.min() >= 0 and k.max() < 96, "unexpected GCP layout"
    t = d.iloc[:, 3].to_numpy(np.float32); ok = t != -9999
    iy0 = int(iy.min()); A = np.full((iy.max() - iy0 + 1, 720, 96), np.nan, np.float32); A[(iy - iy0)[ok], ix[ok], k[ok]] = t[ok]
    n = np.isfinite(A).sum(-1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN slices are the honest gaps
        t1030 = np.nanmean(A[..., [41, 42]], -1); t2230 = np.nanmean(A[..., [89, 90]], -1)  # 10.5 h and 22.5 h sit between bins 41/42 and 89/90
    mean = np.where(n > 0, np.nansum(A, -1) / np.maximum(n, 1), np.nan); frac = np.where(n > 0, (A < 273.15).sum(-1) / np.maximum(n, 1), np.nan)
    return iy0, np.stack([t1030, t2230, mean, frac]).astype(np.float32)

def diviner_lt_grid():
    """All GCP bands -> (4, 360, 720) grid (see gcp_reduce). Each band: download to WORK, reduce, checkpoint OUT/gcp_lt/<band>.npz, delete the table.
    With FETCH_DIVINER False, bands without a checkpoint stay NaN (warning)."""
    ck = OUT / "gcp_lt"; ck.mkdir(exist_ok=True); grid = np.full((4, 360, 720), np.nan, np.float32)
    for band in GCP_BANDS:
        f = ck / f"{band}.npz"
        if not f.exists():
            if not FETCH_DIVINER: warnings.warn(f"Diviner band {band} not reduced and FETCH_DIVINER is False: thermal NaN there"); continue
            tab = fetch(f"gcp_{band}", GCP_URL.format(band), GCP_SHA[band], WORK / f"{band}.tab")
            iy0, r = gcp_reduce(tab); np.savez_compressed(f, iy0=iy0, r=r); tab.unlink()
        z = np.load(f); grid[:, int(z["iy0"]):int(z["iy0"]) + z["r"].shape[1]] = z["r"]
    return grid

def win3(g):
    """NaN-mean over the 3x3 neighbouring 0.5 deg bins (about the 47 km disc). ponytail: lon bins shrink poleward, so the window is narrower there; upgrade = distance-weighted."""
    p = np.pad(g, ((0, 0), (1, 1), (1, 1)), constant_values=np.nan); p[:, :, 0], p[:, :, -1] = p[:, :, -2], p[:, :, 1]  # wrap longitude
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean([p[:, i:i + 360, j:j + 720] for i in range(3) for j in range(3)], axis=0)

def moon_lith_from_feo(feo):
    """PROXY. FeO wt% -> mafic vs feldspathic fraction (linear 6-16 wt%); other classes 0."""
    m = np.clip((np.asarray(feo, float) - 6) / 10, 0, 1)
    return pd.DataFrame(0.0, index=range(len(m)), columns=LITH_COLS).assign(lith_mafic_volcanic=m, lith_felsic_feldspathic=1 - m)

# Mars geologic unit suffix (age prefix e/m/l and A/H/N letters stripped) -> ontology fractions. PROPOSAL for human review, not a published crosswalk:
# volcanic/impact/dune sand -> basaltic crust; highland = basaltic crust + sedimentary cover; transition/lowland/basin/apron/outflow = sedimentary; polar = ice-rich layered deposits.
MARS_ONTO = {**dict.fromkeys(["v", "ve", "vf", "i", "pd"], [1, 0, 0, 0, 0, 0]), **dict.fromkeys(["h", "hu", "he", "hm"], [.7, 0, 0, .3, 0, 0]),
             **dict.fromkeys(["t", "tu"], [.4, 0, 0, .6, 0, 0]), "to": [0, 0, 0, 1, 0, 0], "l": [.5, 0, 0, .5, 0, 0], "b": [.3, 0, 0, .7, 0, 0],
             "a": [0, 0, 0, 1, 0, 0], **dict.fromkeys(["p", "pu", "pe"], [0, 0, 0, .3, 0, .7]), "pc": [0, 0, 0, 0, 0, 1]}
mars_suffix = lambda u: re.sub(r"^[eml]?(AH|HN|AN|A|H|N)", "", u)
mars_proposal = lambda units: pd.DataFrame({u: MARS_ONTO.get(mars_suffix(u), [0] * 6) for u in units}, index=O).T  # unknown suffix -> all 0 -> NaN lithology

def disc_frame(ids, lat, lon, feats, lith, vf_dem):
    """C2 disc table: every feature float32 (NaN where the body has no source); aridity 0.0 by definition (profiles.yaml assumption)."""
    df = pd.DataFrame({"disc_id": np.asarray(ids, np.int32), "lat": lat, "lon": lon})
    for name in EXPECTED_FEATURES: df[name] = np.asarray(feats.get(name, np.full(len(df), np.nan)), np.float32)
    df["aridity_index"] = np.zeros(len(df), np.float32)
    for col in LITH_COLS: df[col] = lith[col].to_numpy(np.float32)
    df["vf_dem"] = np.asarray(vf_dem, np.float32)
    return df

In [ ]:
def moon_discs_real(lat, lon, ids):
    dem = np.fromfile(get("ldem"), "<i2").reshape(11520, 23040)  # 530 MB read once (Drive random access is slow); col 0 = 0 deg E
    t = pd.DataFrame([disc_terrain(dem, 64, la, lo, "moon", lon0=0.0, scale=0.5) for la, lo in zip(lat, lon)]); del dem
    f = {c: t[c].to_numpy() for c in EXPECTED_FEATURES[:4]}
    # Diviner: the 0.5 deg bin containing the disc centre, NaN-mean over the 3x3 bins around it
    g = win3(diviner_lt_grid())[:, np.clip(((90 - lat) / 0.5).astype(int), 0, 359), np.clip(((lon + 180) / 0.5).astype(int), 0, 719)]
    f["lst_mean_K"], f["diurnal_range_K"], f["frac_below_0C"] = g[2], g[0] - g[1], g[3]
    # LP GRS FeO (equal-area pixels, nearest pixel centre; ~5 deg true resolution). Columns: 1-4 = min/max lat/lon, 12 = W_FEO (g/g)
    p = pd.read_csv(get("lpgrs"), sep=r"\s+", header=None, usecols=list(range(13)))
    from scipy.spatial import cKDTree
    _, j = cKDTree(xyz((p[1] + p[2]) / 2, (p[3] + p[4]) / 2)).query(xyz(lat, lon))
    lith = moon_lith_from_feo(100 * p[12].to_numpy()[j])
    return disc_frame(ids, lat, lon, f, lith, t["vf_dem"])

In [ ]:
def mars_discs_real(lat, lon, ids):
    import rasterio, geopandas as gpd
    with rasterio.open(get("mola")) as ds: dem = ds.read(1)  # 2.1 GB int16, metres above the areoid; col 0 = -180 deg E
    t = pd.DataFrame([disc_terrain(dem, 128, la, lo, "mars", lon0=-180.0, scale=1.0) for la, lo in zip(lat, lon)]); del dem
    f = {c: t[c].to_numpy() for c in EXPECTED_FEATURES[:4]}  # thermal and permafrost stay NaN: no temperature product (TES TI is not converted)
    # TES thermal inertia side columns: 3600x7200 big-endian int16 after a 7200 B header, stored south-up/west-lon; [::-1, ::-1] gives north-up, 0..360 E at 0.05 deg
    ti = np.fromfile(get("tes_ti"), ">i2", offset=7200).reshape(3600, 7200)[::-1, ::-1]
    mk = np.fromfile(get("tes_mask"), "u1", offset=7200).reshape(3600, 7200)[::-1, ::-1]  # 1 = TES data, 0 = no data or interpolated
    r, c = np.clip(((90 - lat) / 0.05).astype(int), 0, 3599), np.clip(((lon % 360) / 0.05).astype(int), 0, 7199)
    v = ti[r, c].astype(np.float32); ok = (v >= 24) & (v <= 800)  # 0 = nodata
    # Tanaka 2014 unit at the centre (geographic degrees, same sphere); first match if polygons touch
    geo = gpd.read_file(mars_shapefile()); pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(np.clip(lon, -179.99, 179.99), lat), crs=geo.crs)  # polygon edges stop up to ~0.01 deg short of +-180
    unit = gpd.sjoin(pts, geo[["Unit", "geometry"]], how="left", predicate="intersects")["Unit"]
    unit = unit[~unit.index.duplicated()].reindex(pts.index)
    lith = crosswalk("mars_tanaka2014", unit, mars_proposal(geo["Unit"].unique()))
    out = disc_frame(ids, lat, lon, f, lith, t["vf_dem"])
    out["tes_ti"] = np.where(ok, v, np.nan).astype(np.float32); out["tes_ti_measured"] = np.where(ok, mk[r, c], np.nan).astype(np.float32)
    out["geo_unit"] = unit.to_numpy()  # kept so the proposal can be reviewed unit by unit
    return out

def targets_real(moon, mars):
    """Target features = the nearest disc to each landmark in config/landmarks.yaml (human-owned). While that file is empty: the nearest real disc to the synthetic target
    positions, ids prefixed placeholder_. ponytail: the nearest disc is up to about 60 km (Moon) / 120 km (Mars) from the point; upgrade = run the disc builders at the landmark."""
    lm = load_landmarks()
    if not len(lm):
        t = targets_table(moon, mars)
        return t.assign(target_id=t["target_id"].str.replace("synthetic_", "placeholder_"), name="PLACEHOLDER (nearest real disc) " + t["name"].str.replace("Synthetic ", ""))
    cols = [*EXPECTED_FEATURES, *LITH_COLS]; rows = []
    for r in lm.itertuples():
        d = moon if r.body == "moon" else mars
        disc = d.iloc[int(np.argmin(haversine_km(d["lat"].to_numpy(), d["lon"].to_numpy(), r.lat, r.lon)))]
        rows.append({"target_id": r.id, "name": r.name, "body": r.body, "lat": r.lat, "lon": r.lon, **{c: disc[c] for c in cols}})
    return pd.DataFrame(rows).astype({c: np.float32 for c in cols})

## 5. Assemble and write the C2 tables

Offline (`REAL_DATA = False`): synthetic tables from `make_synthetic` (seeded, fake, ids prefixed `synthetic_`). Real: Earth = Earth Engine reducers + local files, Moon and Mars discs from the builders above, targets from `targets_real`. `vf_dem` / `vf_lst` / `vf_lith` for Earth are 1 where the feature is finite, else 0 (`ponytail:` binary; upgrade = a valid-pixel mean band). Either way the result is checked against the synthetic schema and written to `OUT` together with `tables_source.txt` and the GLiM proposal table that notebook 02 reads.

In [ ]:
def earth_table_real(cells):
    e = earth_remote(cells).merge(earth_local(cells), on="h3", how="left")
    e["vf_dem"] = e["slope_p50_deg"].notna(); e["vf_lst"] = e["lst_mean_K"].notna(); e["vf_lith"] = e["lith_mafic_volcanic"].notna()
    m = prof.params["mask"]  # keep hex if barren+ice >= min OR ndvi <= max; NaN cover/NDVI counts as masked
    e["masked"] = ~(((e["barren_frac"] + e["snow_ice_frac"]) >= m["barren_plus_ice_min"]) | (e["ndvi_mean"] <= m["ndvi_max"]))
    e["tag_periglacial"] = e["frac_below_0C"].fillna(0) > 0.5; e["tag_evaporitic"] = e["lith_evaporite"].fillna(0) > 0.3
    num = [*EXPECTED_FEATURES, *LITH_COLS, "barren_frac", "snow_ice_frac", "ndvi_mean", "aster_mafic_idx", "land_frac", "vf_dem", "vf_lst", "vf_lith"]
    e[num] = e[num].astype(np.float32)
    return e[["h3", *num, "tag_volcanic", "tag_periglacial", "tag_evaporitic", "masked", "lat", "lng"]]

if REAL_DATA:
    cells = select_land_cells(); lat, lon = fibonacci_lattice(N_DISCS); ids = np.arange(N_DISCS)
    if N_DISCS_LIMIT:
        cells = cells[::max(1, len(cells) // N_DISCS_LIMIT)][:N_DISCS_LIMIT]
        sel = np.linspace(0, N_DISCS - 1, N_DISCS_LIMIT).astype(int); lat, lon, ids = lat[sel], lon[sel], ids[sel]
    earth, moon, mars = earth_table_real(cells), moon_discs_real(lat, lon, ids), mars_discs_real(lat, lon, ids)
    TABLES = {"earth_features": earth, "moon_discs": moon, "mars_discs": mars, "targets": targets_real(moon, mars)}
else:
    syn = make_synthetic(SEED, n_hex=6000)
    TABLES = {k: syn[k] for k in ("earth_features", "moon_discs", "mars_discs", "targets")}
need = {"earth_features": ["h3", *EXPECTED_FEATURES, *LITH_COLS, "barren_frac", "snow_ice_frac", "ndvi_mean", "aster_mafic_idx", "land_frac", "vf_dem", "vf_lst", "vf_lith"],
        "moon_discs": ["disc_id", "lat", "lon", *EXPECTED_FEATURES, *LITH_COLS, "vf_dem"],
        "mars_discs": ["disc_id", "lat", "lon", *EXPECTED_FEATURES, *LITH_COLS, "vf_dem"],
        "targets": ["target_id", "name", "body", "lat", "lon", *EXPECTED_FEATURES, *LITH_COLS]}
for k, cols in need.items():
    assert set(cols) <= set(TABLES[k].columns), k
    assert all(TABLES[k][c].dtype == np.float32 for c in [*EXPECTED_FEATURES, *LITH_COLS]), k
    TABLES[k].to_parquet(OUT / f"{k}.parquet", index=False)
(OUT / "tables_source.txt").write_text("real" if REAL_DATA else "synthetic")
pd.DataFrame(GLIM, index=O).T.assign(weight=pd.Series(GLIM_W)).rename_axis("code").to_csv(OUT / "glim_ontology_proposal.csv")  # read by notebook 02 section 4
pd.DataFrame({k: [len(v)] for k, v in TABLES.items()}, index=["rows"])

In [ ]:
# empty feature columns per body (honest gaps, not bugs)
pd.DataFrame({k: TABLES[k][list(EXPECTED_FEATURES)].isna().mean().round(2) for k in ("earth_features", "moon_discs", "mars_discs")})

In [ ]:
# Mars: geologic unit -> six-class PROPOSAL as applied to the discs (for human review; empty offline)
m = TABLES["mars_discs"]
m[["geo_unit", *LITH_COLS]].drop_duplicates("geo_unit").sort_values("geo_unit").set_index("geo_unit").round(2) if "geo_unit" in m else "synthetic run: no Mars unit table"